In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import linear # type: ignore
import mse # type: ignore
from approx import approx # type: ignore
from common import repeat, test_checkgrad_2, test_compare_2 # type: ignore

In [ ]:

class Per_Lin_MSE_Autograd(nn.Module):
    """The version relying fully on PyTorch to handle `forward` and `backward` passes."""
        
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)

    def forward(self, x):
        z = self.model(x)
        return z
    
    def model(self, x):
        z = self.linear(x)
        return z

    def loss(self, p, y):
        L = nn.MSELoss(reduction='mean')(p, y)
        return L

    def predict(self, x):
        with tr.no_grad():
            z = self.model(x)
            return z

In [ ]:

class Per_Lin_MSE_Backward(nn.Module):
    """
    The version where the `forward` and `backward` passes for each operation are written manually, 
    but PyTorch's autograd still orchestrates the overall gradient flow through the computational graph.
    """
    
    def __init__(self, in_features: int, out_features: int, W: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()
        self.linear =  linear.Linear(in_features, out_features, W, b)

    def forward(self, x):
        z = self.model(x)
        return z
    
    def model(self, x):
        z = self.linear(x)
        return z
    
    def loss(self, p, y):
        L = mse.MeanSquaredError()(p, y)
        return L

    def predict(self, x):
        with tr.no_grad():
            z = self.model(x)
            return z    


$$ z(\mathbf{w}, \mathbf{b}) = X\mathbf{w} + \mathbf{b}  \quad $$

$$ X \in R^{n \times m}, \quad \mathbf{w} \in R^{m}, \quad \mathbf{b} \in R^{n}, \quad \mathbf{z} \in R^{n} $$


$$ \frac{\partial z}{\partial \mathbf{w}} = X $$


$$ \frac{\partial z}{\partial \mathbf{b}} = \mathbf{I}_{n} $$

$$ dz = \frac{\partial z}{\partial \mathbf{w}} d\mathbf{w} + \frac{\partial z}{\partial \mathbf{b}} d\mathbf{b} $$

$$ dz = X d\mathbf{w} + \mathbf{I}_{n} \, d\mathbf{b} $$


$$ L(\mathbf{z}) = \frac{1}{N} || \mathbf{z} - \mathbf{y} ||^2 $$

$$ \mathbf{z} \in R^{n}, \quad \mathbf{y} \in R^{n} $$

$$ \frac{dL}{d\mathbf{z}} = 2\frac{1}{N} (\mathbf{z} - \mathbf{y}) $$

$$ dL = \frac{dL}{dz} dz $$

$$ dL = \frac{2}{N} \left \langle \mathbf{z} - \mathbf{y}, d \mathbf{z} \right \rangle $$



$$ \text{Frobenius equality} $$

$$
dL = \left \langle \frac{\partial L}{\partial \mathbf{w}}, d \mathbf{w} \right \rangle _F +
\left \langle \frac{\partial L}{\partial \mathbf{b}}, d \mathbf{b} \right \rangle _F = 
\left \langle \frac{\partial L}{\partial z}, dz \right \rangle _F
$$

$$ \\[2em] $$

$$ 
\left \langle \frac{\partial L}{\partial z}, dz \right \rangle _F =
\left \langle \frac{\partial L}{\partial z}, \frac{\partial z}{\partial \mathbf{w}} d\mathbf{w} + 
\frac{\partial z}{\partial \mathbf{b}} d\mathbf{b} \right \rangle _F =
$$


$$ 
\left \langle \frac{\partial L}{\partial z}, \frac{\partial z}{\partial \mathbf{w}} d\mathbf{w} \right \rangle _F  + 
\left \langle \frac{\partial L}{\partial z}, \frac{\partial z}{\partial \mathbf{b}} d\mathbf{b} \right \rangle _F =
$$


$$ 
\left \langle \Bigg( \frac{\partial z}{\partial \mathbf{w}} \Bigg)^\top \frac{\partial L}{\partial z}, d\mathbf{w} \right \rangle _F  + 
\left \langle \Bigg( \frac{\partial z}{\partial \mathbf{b}} \Bigg)^\top \frac{\partial L}{\partial z}, d\mathbf{b} \right \rangle _F \implies
$$

$$ 
\begin{dcases}
\frac{\partial L}{\partial \mathbf{w}} = \Bigg( \frac{\partial z}{\partial \mathbf{w}} \Bigg)^\top \frac{\partial L}{\partial z}  = X^\top \cdot \frac{\partial L}{\partial z} \\
\\
\frac{\partial L}{\partial \mathbf{b}} = \Bigg( \frac{\partial z}{\partial \mathbf{b}} \Bigg)^\top \frac{\partial L}{\partial z} = \frac{\partial L}{\partial z}
\end{dcases}
$$

$$ \text{Frobenius equality} $$

$$ dF = \left \langle \frac{dF}{dw}, dw \right \rangle + \left \langle \frac{dF}{db}, db \right \rangle = \left \langle \frac{dF}{dL}, dL \right \rangle $$

$$ \\[2em] $$

$$ \left \langle \frac{dF}{dL}, dL \right \rangle = \frac{dF}{dL} \left \langle \frac{dL}{dz}, dz \right \rangle =  $$

$$ 
\frac{dF}{dL} \left \langle \frac{dL}{dz}, \frac{\partial z}{\partial \mathbf{w}} d\mathbf{w} + \frac{\partial z}{\partial \mathbf{b}} d\mathbf{b} \right \rangle = 
$$

$$ 
\frac{dF}{dL} \left \langle \frac{dL}{dz}, \frac{\partial z}{\partial \mathbf{w}} d\mathbf{w} \right \rangle + 
\frac{dF}{dL} \left \langle \frac{dL}{dz},\frac{\partial z}{\partial \mathbf{b}} d\mathbf{b} \right \rangle = 
$$

$$ 
\frac{dF}{dL} \left \langle \Bigg( \frac{\partial z}{\partial \mathbf{w}} \Bigg)^\top \frac{dL}{dz}, d\mathbf{w} \right \rangle + 
\frac{dF}{dL} \left \langle \Bigg( \frac{\partial z}{\partial \mathbf{b}} \Bigg)^\top \frac{dL}{dz}, d\mathbf{b} \right \rangle = 
$$

$$ 
\left \langle \frac{dF}{dL} \Bigg( \frac{\partial z}{\partial \mathbf{w}} \Bigg)^\top \frac{dL}{dz}, d\mathbf{w} \right \rangle + 
\left \langle \frac{dF}{dL} \Bigg( \frac{\partial z}{\partial \mathbf{b}} \Bigg)^\top \frac{dL}{dz}, d\mathbf{b} \right \rangle \implies 
$$

$$ 
\begin{dcases}
\frac{dF}{dw} = \frac{dF}{dL} \Bigg( \frac{\partial z}{\partial \mathbf{w}} \Bigg)^\top \frac{dL}{dz} = 1 \cdot X^\top \cdot \frac{dL}{dz} \\
\\
\frac{dF}{db} = \frac{dF}{dL} \Bigg( \frac{\partial z}{\partial \mathbf{b}} \Bigg)^\top \frac{dL}{dz} = 1 \cdot I_n \cdot \frac{dL}{dz}
\end{dcases}
$$


In [ ]:
class Per_Lin_MSE_Gradient_Function(autograd.Function):
    @staticmethod
    def _model(X, w, b):
        z = linear.linear(X, w, b)
        return z
    
    @staticmethod
    def _loss(p, y):
        L = mse.MeanSquaredError()(p, y)
        return L

    @staticmethod
    def predict(X, w, b):
        with tr.no_grad():
            z = Per_Lin_MSE_Gradient_Function._model(X, w, b)
            return z

    @staticmethod
    def forward(ctx, X, w, b, y):
        z = Per_Lin_MSE_Gradient_Function._model(X, w, b)
        L = Per_Lin_MSE_Gradient_Function._loss(z, y)

        ctx.save_for_backward(X, w, z, y)

        return L
    
    @staticmethod
    def backward(ctx, dF_df):
        (X, w, z, y) = ctx.saved_tensors

        S = X.shape[0]  # Samples
        FI = X.shape[1] # Features In
        FO = w.shape[1] # Features Out
        N = S * FO

        assert X.shape == (S, FI)
        assert w.shape == (FI, FO)
        assert y.shape == (S, FO)
        assert z.shape == (S, FO)
        
        dz_dw = X
        assert dz_dw.shape == (S, FI)

        dL_dz = 2 / N * (z - y)
        assert dL_dz.shape == (S, FO)

        dF_dw = dF_df * dz_dw.t() @ dL_dz
        assert dF_dw.shape == (FI, FO)

        dF_db = dF_df * dL_dz.sum(dim=0)
        assert dF_db.shape == (FO, )

        return (None, dF_dw, dF_db, None)


class Per_Lin_MSE_Gradient(nn.Module):
    """
    The version exposing the exact mechanics of gradient computation and giving 
    full control over how the model participates in PyTorch's autograd system.
    """

    class _Linear:
        """Helper class to keep the model internal layout consistent across all variants."""

        def __init__(self, w, b):
            self.weight = w
            self.bias = b

    # This is helper for testing to indicate that the `forward` method expects both, `x` and `y`.
    CUSTOM_GRAD=True

    def __init__(self, in_features: int, out_features: int, W: tr.Tensor = None, b: tr.Tensor = None) -> None:
        super().__init__()

        # `W` has shape (out_features, in_features) to be consistent with `nn.Linear`
        if W is None:
            self.weight = nn.Parameter(tr.randn(in_features, out_features, dtype=tr.float32))
        else:
            self.weight = nn.Parameter(W.clone().detach().requires_grad_(True))

        if b is None:
            self.bias = nn.Parameter(tr.randn(out_features, dtype=tr.float32))
        else:
            self.bias = nn.Parameter(b.clone().detach().requires_grad_(True))

        self.linear = Per_Lin_MSE_Gradient._Linear(self.weight, self.bias)

    def model(self, x):
        with tr.no_grad():
            z = Per_Lin_MSE_Gradient_Function._model(x, self.weight.t(), self.bias)
            return z
    
    def loss(self, p, y):
        with tr.no_grad():
            L = Per_Lin_MSE_Gradient_Function._loss(p, y)
            return L

    def predict(self, x):
        with tr.no_grad():
            z = Per_Lin_MSE_Gradient_Function.predict(x, self.weight.t(), self.bias)
            return z
        
    def forward(self, x, y):
        z = Per_Lin_MSE_Gradient_Function.apply(x, self.weight.t(), self.bias, y)
        return z

In [ ]:
if __name__ == "__main__":
    test_checkgrad_2(Per_Lin_MSE_Gradient_Function, 1, 1, 1)
    test_checkgrad_2(Per_Lin_MSE_Gradient_Function, 10, 1, 1)
    test_checkgrad_2(Per_Lin_MSE_Gradient_Function, 10, 20, 1)
    test_checkgrad_2(Per_Lin_MSE_Gradient_Function, 10, 20, 30)

    test_compare_2(Per_Lin_MSE_Autograd, Per_Lin_MSE_Backward, 1, 1, 1)
    test_compare_2(Per_Lin_MSE_Autograd, Per_Lin_MSE_Backward, 10, 1, 1)
    test_compare_2(Per_Lin_MSE_Autograd, Per_Lin_MSE_Backward, 10, 20, 1)
    test_compare_2(Per_Lin_MSE_Autograd, Per_Lin_MSE_Backward, 10, 20, 30)

    test_compare_2(Per_Lin_MSE_Autograd, Per_Lin_MSE_Gradient, 1, 1, 1)
    test_compare_2(Per_Lin_MSE_Autograd, Per_Lin_MSE_Gradient, 10, 1, 1)
    test_compare_2(Per_Lin_MSE_Autograd, Per_Lin_MSE_Gradient, 10, 20, 1)
    test_compare_2(Per_Lin_MSE_Autograd, Per_Lin_MSE_Gradient, 10, 20, 30)